# 📘 Notebook 17: Opponents and Self-Play

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module E: Beyond a Single Agent · Notebook 2 of 3 in this module · (17 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what changes when part of the environment is an **opponent**, and why a learning opponent is a moving target
- Train an agent against a **fixed opponent** and show, with an exact calculation, that it remains beatable
- Implement **self-play** and show that it leads to play that cannot be beaten in a small game
- Explain why some games have no good deterministic policy, using rock-paper-scissors
- Run a **tournament** between differently trained agents in a turn-based battle game
- Describe the risks of self-play and why strong systems train against a **pool** of opponents

> **Prerequisites:** Notebook 05 (Q-learning) and Notebook 16 (the game interface, tic-tac-toe, and solving a game by exhaustive search).
>
> 🔭 **Why this notebook matters later:** self-play is the training method behind the strongest game-playing programs, and it is the last missing piece of the AlphaZero loop described in Notebook 16. It also completes a theme that started in Notebook 01: what an agent learns depends on what it is trained **against**.

> 📦 **Libraries used in this notebook.** NumPy, matplotlib, **imageio** and **Pillow** (an image library that Colab already has, used to draw the battle screen). There are no neural networks. All cells together run in about a minute.

In [ ]:
%pip install -q imageio imageio-ffmpeg pillow

In [ ]:
import base64
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import imageio
from PIL import Image, ImageDraw
from IPython.display import HTML, display


def show_video(frames, fps=2, width=360):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. When the environment fights back

### Intuition first
You can learn to juggle alone, because the balls obey the same laws every day. You cannot learn chess alone in the same way. How good a move is depends on what the other player does next, and the other player is trying to make your moves turn out badly.

### Formal definition
In a **two-player zero-sum game**, one player's gain is the other's loss: the rewards of the two always add up to zero. From the point of view of one agent, the opponent is simply part of the environment, and that leads to two very different situations.

- **A fixed opponent** always follows the same policy. The environment, opponent included, is then an ordinary MDP, and every method of this course applies unchanged.
- **A learning opponent** changes its policy over time. The environment is no longer stationary. What was a good move yesterday may be a bad one today, as with the drifting slot machines of Notebook 01.

### Why it matters
An agent learns what it is rewarded for. Against a fixed opponent that is: **beating that opponent**, which is not the same thing as playing the game well. The next two sections measure the difference exactly.

## 2. Learning against a fixed opponent

We use tic-tac-toe, because Notebook 16 showed that it can be solved completely, which gives us a perfect player to measure against. The next cell repeats the game class and the exhaustive solver from there.

In [ ]:
class TicTacToe:
    """State: (board as a tuple of 9 cells, player to move). Cells are 0 (empty), 1 or 2."""
    LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]
    def start(self): return ((0,) * 9, 1)
    def moves(self, state): return [i for i, cell in enumerate(state[0]) if cell == 0]
    def play(self, state, move):
        board, player = state; board = list(board); board[move] = player
        return (tuple(board), 3 - player)
    def winner(self, state):
        """1 or 2 if that player has won, 0 for a draw, None if the game is not over."""
        board = state[0]
        for a, b, c in self.LINES:
            if board[a] and board[a] == board[b] == board[c]: return board[a]
        return 0 if all(board) else None


game = TicTacToe()


def solve(game):
    """Value and best move of every reachable position (Notebook 16)."""
    table = {}

    def value(state):
        if state in table:
            return table[state][0]
        result = game.winner(state)
        if result is not None:
            v = 0 if result == 0 else (1 if result == state[1] else -1)
            table[state] = (v, None)
            return v
        best_value, best_move = -2, None
        for move in game.moves(state):
            v = -value(game.play(state, move))
            if v > best_value:
                best_value, best_move = v, move
        table[state] = (best_value, best_move)
        return best_value

    value(game.start())
    return table


perfect_table = solve(game)
rng = random.Random(0)


def random_player(state):
    return rng.choice(game.moves(state))


def perfect_player(state):
    return perfect_table[state][1]


def match(player_a, player_b, games):
    """Play `games` games, alternating who starts. Returns wins of A, wins of B, draws."""
    score = {"A": 0, "B": 0, "draw": 0}
    for i in range(games):
        players = {1: player_a, 2: player_b} if i % 2 == 0 else {1: player_b, 2: player_a}
        state = game.start()
        while game.winner(state) is None:
            state = game.play(state, players[state[1]](state))
        result = game.winner(state)
        if result == 0:
            score["draw"] += 1
        else:
            score["A" if players[result] is player_a else "B"] += 1
    return score

The learner is the tabular Q-learning agent of Notebook 05. A state is a board together with the player to move, and there is one value for each of the nine squares. The reward comes at the end of the game: +1 for a win, −1 for a loss, 0 for a draw.

In [ ]:
class Learner:
    """Tabular Q-learning for a board game. Only legal moves are ever considered."""

    def __init__(self, seed=0):
        self.Q = {}
        self.rng = random.Random(seed)

    def values(self, state):
        if state not in self.Q:
            self.Q[state] = np.zeros(9)
        return self.Q[state]

    def best_value(self, state):
        values = self.values(state)
        return max(values[move] for move in game.moves(state))

    def act(self, state, epsilon=0.0):
        moves = game.moves(state)
        if self.rng.random() < epsilon:
            return moves[self.rng.randrange(len(moves))]
        values = self.values(state)
        best = max(values[move] for move in moves)
        return self.rng.choice([move for move in moves if values[move] == best])

    def policy(self):
        return lambda state: self.act(state)


def outcome(player, result):
    """The final reward for `player` given the result of the game."""
    return 0.0 if result == 0 else (1.0 if result == player else -1.0)


def train_against(agent, opponent, games, alpha=0.2):
    """Q-learning against a fixed opponent. The opponent's reply is part of the environment."""
    for g in range(games):
        epsilon = max(0.05, 1 - g / (0.6 * games))
        me = 1 + g % 2                                    # play first and second in turn
        state, previous = game.start(), None
        while True:
            if state[1] == me:
                if previous is not None:                  # my earlier move led, via the reply, to this state
                    s, a = previous
                    agent.values(s)[a] += alpha * (agent.best_value(state) - agent.values(s)[a])
                action = agent.act(state, epsilon)
                previous = (state, action)
                state = game.play(state, action)
            else:
                state = game.play(state, opponent(state))
            result = game.winner(state)
            if result is not None:
                if previous is not None:
                    s, a = previous
                    agent.values(s)[a] += alpha * (outcome(me, result) - agent.values(s)[a])
                break
    return agent


start = time.time()
specialist = train_against(Learner(seed=1), random_player, 100000)
print(f"trained for 100,000 games against a random player in {time.time() - start:.0f} seconds")
print("against the random player, 1,000 games: ", match(specialist.policy(), random_player, 1000))
print("against the perfect player, 100 games:  ", match(specialist.policy(), perfect_player, 100), "  (A is the learner)")

Against the opponent it trained with, the agent is excellent: it wins the great majority of games and almost never loses. Against the perfect player it may well draw every game. Is it then a perfect player itself?

Sample games cannot answer that. Our perfect player always chooses the same best move in a given position, so a hundred games against it test the same few lines of play again and again. The agent might have serious weaknesses that this particular opponent never steers into.

### An exact measurement
Because the game is solved, we do not have to rely on sample games. We can compute exactly how the agent's fixed policy fares against the **best possible reply**, an opponent that looks for its weaknesses, and count in how many positions it makes a move that gives away a win or a draw.

In [ ]:
def worst_case(policy, me):
    """The result for `me` if it follows `policy` and the opponent always finds the best reply."""
    memo = {}

    def value(state):
        if state in memo:
            return memo[state]
        result = game.winner(state)
        if result is not None:
            v = outcome(me, result)
        elif state[1] == me:
            v = value(game.play(state, policy(state)))
        else:
            v = min(value(game.play(state, move)) for move in game.moves(state))
        memo[state] = v
        return v

    return value(game.start())


def count_mistakes(agent):
    """In how many positions does the agent's move throw away value, compared with perfect play?"""
    mistakes = positions = 0
    for state, (value, best_move) in perfect_table.items():
        if best_move is None:
            continue                                        # the game is already over
        positions += 1
        move = agent.act(state)
        if -perfect_table[game.play(state, move)][0] < value:
            mistakes += 1
    return mistakes, positions


def report(name, agent):
    names = {1.0: "wins", 0.0: "draws", -1.0: "LOSES"}
    mistakes, positions = count_mistakes(agent)
    print(f"{name}: against the best possible opponent it {names[worst_case(agent.policy(), 1)]} when moving first "
          f"and {names[worst_case(agent.policy(), 2)]} when moving second; "
          f"mistakes in {mistakes} of {positions} positions")


report("trained against random", specialist)

The agent **can be beaten**, and there are hundreds of positions in which it throws away a win or a draw. It never learned better, for a simple reason: a random opponent rarely punishes a mistake, so mistakes cost almost nothing, and many dangerous positions hardly ever arose. The agent learned to **beat a random player**. It did not learn tic-tac-toe.

## 3. Self-play

### Intuition first
The best sparring partner is one who is about as good as you are and finds your weaknesses. Where do you get one for every stage of your development? You play against **yourself**. Whenever you discover a trick, your opponent knows it too, and you are forced to find the answer to it.

### Formal definition
In **self-play** one agent plays both sides. It needs only one Q-table, because the game looks the same from either seat. After a move from state $S$ to state $S'$, it is the **opponent's** turn in $S'$, and in a zero-sum game whatever is good for the opponent is equally bad for me:

$$Q(S, A) \leftarrow Q(S, A) + \alpha\,\big[\,R - \max_{a'} Q(S', a') - Q(S, A)\,\big]$$

Compare with Q-learning in Notebook 05. The $+\gamma\max$ has become $-\max$: the value of the next state counts **against** the player who just moved. It is the minimax rule of Notebook 16, learned from samples where there it was computed exhaustively.

In [ ]:
def train_self_play(agent, games, alpha=0.2):
    """One agent plays both sides and learns from every move."""
    for g in range(games):
        epsilon = max(0.05, 1 - g / (0.6 * games))
        state = game.start()
        while True:
            action = agent.act(state, epsilon)
            next_state = game.play(state, action)
            result = game.winner(next_state)
            if result is not None:
                target = outcome(state[1], result)              # the game is over: the real result
            else:
                target = -agent.best_value(next_state)          # good for my opponent means bad for me
            agent.values(state)[action] += alpha * (target - agent.values(state)[action])
            if result is not None:
                break
            state = next_state
    return agent


start = time.time()
self_taught = train_self_play(Learner(seed=2), 100000)
print(f"trained for 100,000 games of self-play in {time.time() - start:.0f} seconds")
print("against the random player, 1,000 games: ", match(self_taught.policy(), random_player, 1000))
print("against the perfect player, 100 games:  ", match(self_taught.policy(), perfect_player, 100), "  (A is the learner)\n")

report("trained against random", specialist)
report("trained by self-play   ", self_taught)

The self-taught agent never met the perfect player and never met a random one. Against the best possible opponent it cannot be beaten from either seat, and the positions in which it still makes a mistake are a handful out of several thousand.

Notice also what it does **worse**. Against the random player it wins fewer games than the specialist does. The specialist learned to take risks that a careless opponent will not punish. The self-taught agent plays soundly against everybody and therefore exploits nobody in particular. Which of the two is "better" depends on who is sitting across the board.

In [ ]:
checkpoints = [500, 2000, 5000, 20000, 50000]
history = []
for games in checkpoints:
    agent = train_self_play(Learner(seed=2), games)
    history.append(count_mistakes(agent)[0])
    print(f"after {games:>6,} games of self-play: mistakes in {history[-1]:>4} positions")

plt.figure(figsize=(7, 4))
plt.plot(checkpoints, history, marker="o")
plt.xscale("log")
plt.title("Self-play: positions in which the agent's move is a mistake")
plt.xlabel("games of self-play (logarithmic scale)"); plt.ylabel("number of positions"); plt.grid(True)
plt.show()

## 4. Why self-play works, and where it fails

**Why it works**

- **The opponent is always at the right level.** A beginner facing a master learns nothing but that it loses. A beginner facing a beginner sees the consequences of its own choices, and the difficulty rises exactly as fast as its own ability.
- **Weaknesses are found.** Whatever the agent does badly, its opponent, being itself, is in a position to exploit, and the exploiting gets rewarded.
- **No teacher is needed.** No recorded games, no hand-written opponent, only the rules.

**Where it fails**

- **Shared blind spots.** If neither copy ever plays into some part of the game, neither learns it. Exploration matters as much as ever.
- **Forgetting.** The agent only meets its **current** self. It can forget how to answer a strategy it abandoned long ago, and lose to it when someone else plays it.
- **Chasing its own tail.** In some games every strategy is beaten by another one. Two learners can then go round in circles for ever.

The last point deserves a demonstration, because the simplest example is a game everybody knows.

## 5. Games without a best move

### Intuition first
In rock-paper-scissors, which move is best? None. Any move you always play is beaten by the opponent who notices. The only safe way to play is to be **unpredictable**: choose each move with probability one third. A deterministic policy, the kind that `argmax` produces, is the wrong tool for this game.

Two simple learners play each other. Each keeps an estimate of the value of rock, paper and scissors (a three-armed bandit, as in Notebook 01, with a constant step size) and mostly plays its current favourite.

In [ ]:
PAYOFF = np.array([[0, -1, 1],          # rows: my move (rock, paper, scissors)
                   [1, 0, -1],          # columns: the opponent's move
                   [-1, 1, 0]])         # entries: my reward

rps_rng = np.random.default_rng(0)
Q = [np.zeros(3), np.zeros(3)]
choices = [[], []]

for step in range(3000):
    moves = [int(rps_rng.integers(3)) if rps_rng.random() < 0.1 else int(Q[p].argmax()) for p in (0, 1)]
    for p in (0, 1):
        reward = PAYOFF[moves[p], moves[1 - p]]
        Q[p][moves[p]] += 0.1 * (reward - Q[p][moves[p]])
        choices[p].append(moves[p])

first = np.array(choices[0])
window = 100
plt.figure(figsize=(9, 3.5))
for move, name in enumerate(("rock", "paper", "scissors")):
    share = np.convolve(first == move, np.ones(window) / window, mode="valid")
    plt.plot(share, label=name)
plt.title("Player 1: how often each move was played (moving average of 100 rounds)")
plt.xlabel("round"); plt.ylabel("share of rounds"); plt.grid(True); plt.legend()
plt.show()

print("player 1, share of each move over all 3,000 rounds:", np.bincount(first, minlength=3) / 3000)

The learner never settles. It favours one move for a while, the opponent learns to beat it, it switches to the move that beats the opponent's new favourite, and so on, round and round. Yet look at the last line: **averaged over time**, each move is played about a third of the time, which is the right answer.

A policy that chooses its actions at random with given probabilities is called a **mixed strategy**, and a pair of strategies from which neither player can gain by deviating alone is a **Nash equilibrium**. For games like this one, where the players move at the same time or hide information from each other, value-based methods with `argmax` cannot represent the solution. Policy-based methods can, because their output **is** a probability for each action (Notebook 12). Poker programs and the agents for games such as StarCraft are built on that side of the field.

> ⚠️ **Common pitfalls**
>
> - **Judging an agent only against its training partner.** The specialist of Section 2 looked excellent by that measure.
> - **Believing that beating a weak opponent more often means playing better.** The specialist beat the random player more often than the perfect player would.
> - **Self-play without exploration.** Two greedy copies repeat the same game for ever.
> - **Using `argmax` in a game that needs unpredictability.** The result is a policy that anyone can learn to beat.

## 6. Project: monster battles

Tic-tac-toe is solved, so it cannot surprise us. For the project we build a small game in the style of monster-battling video games and hold a tournament in it.

### The rules

- Each player has a team of three monsters: a **fire**, a **water** and a **grass** monster, with 100 health points each. One of them is active.
- **Fire beats grass, water beats fire, grass beats water.** An attack does about 25 points of damage, **doubled** against the type it beats and **halved** against the type that beats it and against its own type.
- The players take turns. On your turn you either **attack** with your active monster or **switch** to another one. Switching uses up the turn.
- A monster that has just come in must attack once before it can be switched out again.
- When a monster reaches 0 health it faints, and the next one comes in. A player with no monsters left has lost.

The decisions are real ones. Staying in with a monster at a disadvantage is costly, but so is switching, because the opponent gets a free attack.

### What the agent sees
The observation has seven parts: my active monster's type, its health as one of three levels (low, medium, high), whether it has just come in, the opponent's active type and its health level, and which of my monsters and of the opponent's are still standing. As in Snake (Notebook 06), this is a deliberate simplification: the exact health points, and the health of the monsters on the bench, are **not** visible.

In [ ]:
TYPES = ["fire", "water", "grass"]
BEATS = {0: 2, 1: 0, 2: 1}            # fire beats grass, water beats fire, grass beats water


def effectiveness(attacker, defender):
    if BEATS[attacker] == defender:
        return 2.0
    return 0.5 if (BEATS[defender] == attacker or attacker == defender) else 1.0


class Battle:
    """Two teams of three monsters (fire, water, grass) take turns.
    Actions: 0 attack, 1/2/3 switch to the fire/water/grass monster.
    A monster that has just come in must attack once before it can be switched out again."""

    def __init__(self, seed=0, max_turns=80):
        self.rng = random.Random(seed)
        self.max_turns = max_turns

    def reset(self):
        self.hp = [[100.0] * 3, [100.0] * 3]
        self.active = [self.rng.randrange(3), self.rng.randrange(3)]
        self.fresh = [False, False]              # has the active monster just been switched in?
        self.to_move = self.rng.randrange(2)
        self.turn = 0
        return self.observe()

    def observe(self):
        """The situation as the player to move sees it."""
        me, other = self.to_move, 1 - self.to_move
        bucket = lambda hp: 0 if hp <= 33 else 1 if hp <= 66 else 2
        alive = lambda p: tuple(int(h > 0) for h in self.hp[p])
        return (self.active[me], bucket(self.hp[me][self.active[me]]), int(self.fresh[me]),
                self.active[other], bucket(self.hp[other][self.active[other]]), alive(me), alive(other))

    def legal(self):
        me = self.to_move
        moves = [0]
        if not self.fresh[me]:
            moves += [t + 1 for t in range(3) if t != self.active[me] and self.hp[me][t] > 0]
        return moves

    def step(self, action):
        """The player to move acts. Returns (observation for the next player, reward for the mover, done)."""
        me, other = self.to_move, 1 - self.to_move
        if action not in self.legal():
            action = 0
        if action == 0:
            a, d = self.active[me], self.active[other]
            damage = 25 * effectiveness(a, d) * self.rng.uniform(0.85, 1.0)
            self.hp[other][d] = max(0.0, self.hp[other][d] - damage)
            self.fresh[me] = False
            if self.hp[other][d] == 0 and any(h > 0 for h in self.hp[other]):
                self.active[other] = next(t for t in range(3) if self.hp[other][t] > 0)
                self.fresh[other] = False
        else:
            self.active[me] = action - 1
            self.fresh[me] = True
        self.turn += 1
        won = not any(h > 0 for h in self.hp[other])
        done = won or self.turn >= self.max_turns
        self.to_move = other
        return self.observe(), (1.0 if won else 0.0), done

In [ ]:
COLOURS = {0: (225, 90, 50), 1: (60, 130, 225), 2: (80, 170, 90)}


def draw_battle(env, message=""):
    """The battle as a picture: one row per player, a square and a health bar per monster."""
    image = Image.new("RGB", (360, 240), (245, 245, 240))
    draw = ImageDraw.Draw(image)
    for player, top in ((1, 30), (0, 130)):
        draw.text((10, top - 18), "player 2" if player else "player 1", fill=(60, 60, 60))
        for t in range(3):
            x = 20 + t * 115
            fainted = env.hp[player][t] <= 0
            colour = (200, 200, 200) if fainted else COLOURS[t]
            outline = (0, 0, 0) if env.active[player] == t and not fainted else None
            draw.rectangle([x, top, x + 50, top + 50], fill=colour, outline=outline, width=4)
            draw.text((x + 58, top + 4), TYPES[t], fill=(60, 60, 60))
            draw.rectangle([x, top + 58, x + 100, top + 68], fill=(220, 220, 220))
            draw.rectangle([x, top + 58, x + env.hp[player][t], top + 68], fill=(70, 160, 70))
    draw.text((10, 218), message, fill=(30, 30, 30))
    return np.array(image)


def play_battle(env, players, record=False):
    """One battle between two players (functions of observation and legal actions). Returns the winner or None."""
    observation = env.reset()
    frames = [draw_battle(env, "start")] if record else []
    done, winner = False, None
    while not done:
        mover = env.to_move
        action = players[mover](observation, env.legal())
        observation, reward, done = env.step(action)
        if reward > 0:
            winner = mover
        if record:
            what = "attacks" if action == 0 else f"switches to {TYPES[action - 1]}"
            frames.append(draw_battle(env, f"player {mover + 1} {what}"))
    return winner, frames

### Three opponents written by hand

- The **random** player picks any legal action.
- The **attacker** always attacks.
- The **tactician** follows the rule a human would: if my active monster is weak against theirs, switch to the monster that beats theirs if I can, and otherwise attack.

In [ ]:
battle_rng = random.Random(0)


def random_bot(observation, legal):
    return legal[battle_rng.randrange(len(legal))]


def attacker(observation, legal):
    return 0


def tactician(observation, legal):
    mine, my_health, fresh, theirs, their_health, my_alive, their_alive = observation
    if BEATS[theirs] == mine:                                   # my active monster is weak against theirs
        counter = next(t for t in range(3) if BEATS[t] == theirs)
        if counter + 1 in legal:
            return counter + 1                                  # bring in the monster that beats theirs
        if theirs + 1 in legal:
            return theirs + 1                                   # or at least one that resists it
    return 0


def win_rate(player_a, player_b, battles=1000, seed=123):
    """Percentage of battles won by A. Who moves first is decided at random in each battle."""
    env = Battle(seed)
    wins = sum(play_battle(env, [player_a, player_b])[0] == 0 for b in range(battles))
    return 100 * wins / battles


print(f"attacker against random:    {win_rate(attacker, random_bot):3.0f}%")
print(f"tactician against random:   {win_rate(tactician, random_bot):3.0f}%")
print(f"tactician against attacker: {win_rate(tactician, attacker):3.0f}%")

winner, frames = play_battle(Battle(seed=4), [tactician, attacker], record=True)
print(f"\nvideo: tactician (player 1) against attacker (player 2), winner: player {winner + 1}")
show_video(frames)

Knowing the type chart is worth a great deal: the tactician beats the attacker in most battles with nothing more than that one rule.

### Learning agents
The learner is again tabular Q-learning. It trains against a list of opponents, called a **pool**, and faces one of them, chosen at random, in each battle. With a single opponent in the list this is ordinary training against a fixed opponent.

In [ ]:
class BattleLearner:
    def __init__(self, seed=0):
        self.Q = {}
        self.rng = random.Random(seed)

    def values(self, observation):
        if observation not in self.Q:
            self.Q[observation] = np.zeros(4)
        return self.Q[observation]

    def best_value(self, observation, legal):
        values = self.values(observation)
        return max(values[a] for a in legal)

    def act(self, observation, legal, epsilon=0.0):
        if self.rng.random() < epsilon:
            return legal[self.rng.randrange(len(legal))]
        values = self.values(observation)
        best = max(values[a] for a in legal)
        return self.rng.choice([a for a in legal if values[a] == best])

    def policy(self):
        return lambda observation, legal: self.act(observation, legal)


def train_battles(agent, pool, battles, alpha=0.1, gamma=0.95, seed=0):
    """Q-learning as player 1 against opponents drawn from `pool`."""
    env = Battle(seed)
    chooser = random.Random(seed)
    for b in range(battles):
        opponent = pool[chooser.randrange(len(pool))]
        epsilon = max(0.05, 1 - b / (0.6 * battles))
        observation, done, previous = env.reset(), False, None
        while not done:
            if env.to_move == 0:                                        # the learner's turn
                legal = env.legal()
                if previous is not None:
                    o, a = previous
                    agent.values(o)[a] += alpha * (gamma * agent.best_value(observation, legal) - agent.values(o)[a])
                action = agent.act(observation, legal, epsilon)
                previous = (observation, action)
                observation, reward, done = env.step(action)
                if done:                                                # +1 if that move won, 0 for a time-out
                    agent.values(previous[0])[action] += alpha * (reward - agent.values(previous[0])[action])
            else:                                                       # the opponent's turn
                observation, reward, done = env.step(opponent(observation, env.legal()))
                if done and previous is not None:                       # -1 if the opponent has just won
                    o, a = previous
                    agent.values(o)[a] += alpha * ((-1.0 if reward > 0 else 0.0) - agent.values(o)[a])
    return agent


start = time.time()
trained = {
    "trained vs attacker":  train_battles(BattleLearner(1), [attacker], 90000),
    "trained vs tactician": train_battles(BattleLearner(2), [tactician], 90000),
    "trained vs the pool":  train_battles(BattleLearner(3), [random_bot, attacker, tactician], 90000),
}
print(f"three agents trained for 90,000 battles each in {time.time() - start:.0f} seconds")

### The tournament
Every player meets every other, 500 battles per pairing. Each row shows how often that player **won** against the player of each column.

In [ ]:
players = {"random": random_bot, "attacker": attacker, "tactician": tactician}
players.update({name: agent.policy() for name, agent in trained.items()})
names = list(players)

table = np.array([[win_rate(players[a], players[b], battles=500) for b in names] for a in names])

plt.figure(figsize=(8, 5.5))
plt.imshow(table, cmap="RdYlGn", vmin=0, vmax=100)
for i in range(len(names)):
    for j in range(len(names)):
        plt.text(j, i, f"{table[i, j]:.0f}", ha="center", va="center")
plt.xticks(range(len(names)), names, rotation=30, ha="right"); plt.yticks(range(len(names)), names)
plt.title("Tournament: % of battles won by the row player against the column player")
plt.colorbar(shrink=0.8)
plt.show()

print(f"{'':22s}" + "".join(f"{name[-9:]:>11s}" for name in names) + "    average vs hand-written")
for i, name in enumerate(names):
    print(f"{name:22s}" + "".join(f"{table[i, j]:11.0f}" for j in range(len(names))) + f"{table[i, :3].mean():15.1f}")

Read the three rows of the learning agents.

- **Trained against the attacker.** It beats the attacker almost every time, and loses badly to the tactician. It learned to win races of pure damage, and has never seen an opponent who changes monster.
- **Trained against the tactician.** It does much better against the tactician than the first agent, though it does not match it. But look at its result against the plain **attacker**, a far weaker player: it loses more battles than it wins. Specialising against a clever opponent has left it unprepared for a simple one whose behaviour it never met.
- **Trained against the pool.** It has the best record across the board. Variety in training bought robustness.

None of the learners beats the tactician clearly. The reason is the one we met with Snake: the **observation**. The agent sees health in three coarse levels and nothing of the monsters on the bench, so situations that call for different decisions look alike to it. The hand-written rule does not need that information, and no amount of training can supply what the observation withholds.

### Did the agent discover the type chart?
Nobody told the learners which type beats which. Here is what the pool-trained agent does when its active monster is weak against the opponent's, with all monsters healthy.

In [ ]:
agent = trained["trained vs the pool"]
ACTIONS = ["attack", "switch to fire", "switch to water", "switch to grass"]

for mine in range(3):
    theirs = next(t for t in range(3) if BEATS[t] == mine)                 # the type that beats mine
    observation = (mine, 2, 0, theirs, 2, (1, 1, 1), (1, 1, 1))
    legal = [0] + [t + 1 for t in range(3) if t != mine]
    values = agent.values(observation)
    choice = max(legal, key=lambda a: values[a])
    counter = next(t for t in range(3) if BEATS[t] == theirs)
    print(f"my {TYPES[mine]:5s} faces their {TYPES[theirs]:5s}: the agent chooses '{ACTIONS[choice]}'"
          f"   (the type that beats {TYPES[theirs]} is {TYPES[counter]})")

Compare what the agent chooses with the type in brackets. Where they agree, the agent has worked out one line of the type chart from nothing but wins and losses. Where they do not, look again at the rules: switching costs a turn, and the agent may have found that in this particular situation the cost outweighs the gain.

## 7. Beyond one opponent

The tournament points at how the strongest systems are trained. AlphaZero (Notebook 16) used pure self-play, which was enough for chess and Go, where the rules are symmetric and everything is visible. For StarCraft II, a game with hidden information and many viable strategies, DeepMind's AlphaStar trained a whole **league**: a growing population of past and present agents, some of whose only job was to find the weaknesses of the others. A pool of opponents, as in our small experiment, is the same idea in miniature.

The principle to keep is the one this notebook started with. An agent becomes good at what it is trained against. Choosing the opponents is part of designing the learning problem, as much as choosing the reward and the observation.

---
## ✏️ Exercises

The exercises use `game`, `Learner`, `train_against`, `train_self_play`, `match`, `report`, `perfect_player`, and from the battle project `Battle`, `BattleLearner`, `train_battles`, `win_rate`, `play_battle`, the three bots and the `trained` agents.

### Exercise 1 (Catching the specialist out)
`worst_case` says that the specialist of Section 2 loses when it moves second against the best possible opponent. Find the game. Write a function that plays X against the specialist and always chooses the move that leads to the worst outcome for it (the same recursion as in `worst_case`, but returning the move), play that game, and print the board after every move with the `show` function below. Where did the specialist go wrong?

In [ ]:
def show(state):
    symbols = {0: ".", 1: "X", 2: "O"}
    for row in range(3):
        print(" ".join(symbols[state[0][3 * row + col]] for col in range(3)))
    print()

# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def show(state):
    symbols = {0: ".", 1: "X", 2: "O"}
    for row in range(3):
        print(" ".join(symbols[state[0][3 * row + col]] for col in range(3)))
    print()


victim = specialist.policy()
memo = {}

def value_for_victim(state):
    """The outcome for the specialist (playing O) if X always finds its worst reply."""
    if state not in memo:
        result = game.winner(state)
        if result is not None:
            memo[state] = outcome(2, result)
        elif state[1] == 2:
            memo[state] = value_for_victim(game.play(state, victim(state)))
        else:
            memo[state] = min(value_for_victim(game.play(state, move)) for move in game.moves(state))
    return memo[state]


def exploiter(state):
    return min(game.moves(state), key=lambda move: value_for_victim(game.play(state, move)))


state = game.start()
while game.winner(state) is None:
    state = game.play(state, exploiter(state) if state[1] == 1 else victim(state))
    show(state)
print("winner:", {1: "X, the exploiter", 2: "O, the specialist", 0: "nobody"}[game.winner(state)])
```

*Look for the first position in which X threatens two lines at once. The move O made just before it is the mistake: a position that the specialist met rarely against a random opponent, and in which a random opponent would usually have failed to punish it. The perfect player of Section 2 never led the game there, which is why a hundred sample games showed nothing.*
</details>

### Exercise 2 (A lopsided opponent)
A rock-paper-scissors player chooses rock half of the time and paper and scissors a quarter each. Let a bandit learner (as in Section 5, epsilon 0.1, step size 0.1) play 5,000 rounds against it. Which move does the learner end up favouring, and what is its average reward per round? What would the reward be against a player who chooses uniformly at random?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
lopsided_rng = np.random.default_rng(1)
Q_learner = np.zeros(3)
total = 0.0
for step in range(5000):
    opponent_move = int(lopsided_rng.choice(3, p=[0.5, 0.25, 0.25]))
    move = int(lopsided_rng.integers(3)) if lopsided_rng.random() < 0.1 else int(Q_learner.argmax())
    reward = PAYOFF[move, opponent_move]
    Q_learner[move] += 0.1 * (reward - Q_learner[move])
    total += reward

print("estimated values of rock, paper, scissors:", Q_learner.round(2))
print(f"favourite move: {['rock', 'paper', 'scissors'][int(Q_learner.argmax())]},  average reward per round: {total / 5000:+.3f}")
```

*Paper beats rock, so against this opponent paper wins half the time, draws a quarter and loses a quarter: an expected reward of +0.25 per round for a player who always chooses paper. The learner gets somewhat less, because it explores on one round in ten and needed some rounds to find out. Against a uniformly random player every move has an expected reward of exactly 0, and nothing can be gained. A predictable opponent can be exploited. An unpredictable one cannot.*
</details>

### Exercise 3 (How long must self-play run?)
Train self-play agents for 100,000 games with `alpha` = 0.05, 0.2 and 0.8 and report each with `report`. Which step size leaves the fewest mistakes? Tic-tac-toe has no randomness at all: why is a small step size not automatically the best here?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for alpha in (0.05, 0.2, 0.8):
    agent = train_self_play(Learner(seed=2), 100000, alpha=alpha)
    report(f"alpha = {alpha:<4}", agent)
```

*The game is deterministic, so there is no noise in the rewards to average away. But the targets still move, because they are computed from values that are themselves being learned, and with a small step size the corrections travel slowly from the end of the game back to the opening. A larger step spreads them faster. In a noisy game such as the battles, where damage is random, a large step would again be harmful.*
</details>

### Exercise 4 (Enter the tournament)
Write your own battle player as a function `my_bot(observation, legal)` that returns one of the legal actions. The observation is the tuple described in Section 6, and `BEATS[t]` is the type that type `t` beats. Try to improve on the tactician, for example by not switching when the opponent's active monster is nearly finished. Measure your bot against the six players of the tournament.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def my_bot(observation, legal):
    mine, my_health, fresh, theirs, their_health, my_alive, their_alive = observation
    if their_health == 0:
        return 0                                            # their monster is almost finished: finish it
    if BEATS[theirs] == mine:
        counter = next(t for t in range(3) if BEATS[t] == theirs)
        if counter + 1 in legal:
            return counter + 1
        if theirs + 1 in legal and my_health > 0:
            return theirs + 1
    return 0

for name in names:
    print(f"my bot against {name:22s} {win_rate(my_bot, players[name], battles=500):3.0f}%")
```

*This is one possible bot, a tactician that does not walk away from a monster it could finish off. Whether it beats the plain tactician is for the measurement to say: with random damage, 500 battles still leave an uncertainty of a few percentage points, so small differences mean little.*
</details>

### Exercise 5 (Self-play in the battle game (a little harder))
Tic-tac-toe self-play gave an unbeatable agent. Try it in the battle game. Write `train_battle_self_play(agent, battles)`, in which one `BattleLearner` plays both sides and every move is updated with the rule of Section 3 (the target is the reward if the move ends the battle, and otherwise minus `gamma` times the best value of the next observation, which belongs to the opponent). Train for 90,000 battles and measure the agent against the three hand-written opponents. Compare with the pool-trained agent, and suggest why self-play does not repeat its success here.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def train_battle_self_play(agent, battles, alpha=0.1, gamma=0.95, seed=0):
    env = Battle(seed)
    for b in range(battles):
        epsilon = max(0.05, 1 - b / (0.6 * battles))
        observation, done = env.reset(), False
        while not done:
            action = agent.act(observation, env.legal(), epsilon)
            next_observation, reward, done = env.step(action)
            target = reward if done else -gamma * agent.best_value(next_observation, env.legal())
            agent.values(observation)[action] += alpha * (target - agent.values(observation)[action])
            observation = next_observation
    return agent

self_played = train_battle_self_play(BattleLearner(4), 90000)
for name, opponent in (("random", random_bot), ("attacker", attacker), ("tactician", tactician)):
    print(f"against {name:10s} self-play agent {win_rate(self_played.policy(), opponent):3.0f}%    "
          f"pool-trained agent {win_rate(trained['trained vs the pool'].policy(), opponent):3.0f}%")
```

*In our runs the self-play agent was clearly weaker than the pool-trained one, in particular against the attacker. The minus-max rule assumes that the next observation tells the opponent's whole situation, and in tic-tac-toe it does. Here the observation is coarse and hides the benches, so the same observation stands for many different real situations, and the values the two sides pass back and forth are averages over all of them. Against fixed opponents the learner at least faces a stable problem. Self-play removes even that. Self-play is a powerful method for games the agent can see completely, and it is not a cure for an observation that hides what matters.*
</details>

## 🔑 Key takeaways

- Against a **fixed opponent** the environment is an ordinary MDP, and the agent learns to beat **that opponent**, which may be far from playing well.
- In a solved game the weakness can be measured exactly: the specialist made mistakes in hundreds of positions and could be beaten.
- **Self-play** trains one agent against itself with the rule "good for my opponent is bad for me", and reached unbeatable play in tic-tac-toe.
- Some games, like rock-paper-scissors, have **no good deterministic policy**. They need **mixed strategies**, which policy-based methods can represent.
- In the tournament, agents trained against one opponent were **specialists**, and the agent trained against a **pool** was the most robust.
- What an agent is trained **against** is a design decision, alongside the reward and the observation.

---
**Next:** *Notebook 18: Reinforcement Learning in Practice*, on how to run, judge and report experiments like the ones in this course.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*